# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import pandas as pd
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [2]:
import tensorflow as tf
# Cihazları listele ve Metal'i zorla
devices = tf.config.list_physical_devices('GPU')
if devices:
    print("Metal GPU Bulundu!")

Metal GPU Bulundu!


In [3]:
import tensorflow as tf

# GPU kontrolü
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"M4 GPU Aktif: {gpu_devices}")
    # Bellek yönetimini optimize et
    for gpu in gpu_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("M4 GPU bulunamadı, CPU üzerinden çalışacak.")

M4 GPU Aktif: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
df = pd.read_csv("data/spam_Emails_data.csv")
df.head()

,label,text
0,Spam,viiiiiiagraaaa\nonly for the ones that want to...
1,Ham,got ice thought look az original message ice o...
2,Spam,yo ur wom an ne eds an escapenumber in ch ma n...
3,Spam,start increasing your odds of success & live s...
4,Ham,author jra date escapenumber escapenumber esca...


In [7]:
# 1. Veriyi Hazırla
# Varsayalım ki df['text'] mail içeriği, df['label'] ise "spam"/"ham" yazısı
texts = df['text'].values
labels = df['label'].map({'Ham': 0, 'Spam': 1}).values # Manuel eşleme daha güvenlidir

In [8]:
import pandas as pd

# 1. Boş (NaN) değerleri veri setinden tamamen at
df = df.dropna(subset=['text'])

# 2. Her ihtimale karşı tüm metinleri string formatına çevir
# (Bazı satırlarda sadece rakam varsa onlar da float kalmış olabilir)
texts = df['text'].astype(str).values


In [9]:
# 2. Tokenization (Kelimeleri Sayıya Çevir)
max_words = 20000
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

In [10]:
# 3. Padding (Boyutları Eşitle)
max_len = 150 # Her maili 100 kelimeye sabitle
X = pad_sequences(sequences, maxlen=max_len)

In [13]:
from tensorflow.keras.layers import Bidirectional

model = Sequential([
    # 1. Embedding: 128 boyutlu vektör uzayı
    Embedding(input_dim=max_words, output_dim=128),

    # 2. Bidirectional LSTM: Metni hem baştan sona hem sondan başa okur.
    # Bu, "kazandınız" kelimesinin öncesindeki küfürleri veya bağlamı daha iyi anlar.
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.25),

    # 3. İkinci LSTM: Derin özellikleri yakalamak için
    LSTM(64),
    Dropout(0.25),


    # 4. Final: 1'e iniş
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# 5. Eğitim
model.fit(X, labels, epochs=10,batch_size=1024, validation_split=0.2)

Epoch 1/10


2026-03-17 09:21:26.179188: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


152/152 ━━━━━━━━━━━━━━━━━━━━ 253s 2s/step - accuracy: 0.9307 - loss: 0.1699 - val_accuracy: 0.9759 - val_loss: 0.0699
Epoch 2/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 277s 2s/step - accuracy: 0.9833 - loss: 0.0532 - val_accuracy: 0.9838 - val_loss: 0.0508
Epoch 3/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 250s 2s/step - accuracy: 0.9897 - loss: 0.0334 - val_accuracy: 0.9852 - val_loss: 0.0475
Epoch 4/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.9934 - loss: 0.0217 - val_accuracy: 0.9810 - val_loss: 0.0588
Epoch 5/10
105/152 ━━━━━━━━━━━━━━━━━━━━ 1:11 2s/step - accuracy: 0.9948 - loss: 0.0172

In [ ]:
import numpy as np

# 1. Test etmek istediğin rastgele mail(ler)
random_mails = [
    "LOOK HERE YOU SON OF A BITCH! YOU WIN 1000$ FREE SPIN COUPON.",
    "BEST BET SITE HERE, YOU WON 100$ FREE SPIN COUPON", # Spam olması beklenen
# Spam olması beklenen
    "Sarp, see you in the lab. Do not forget to bring your circuit"      # Ham olması beklenen
]

# 2. Metni modelin anlayacağı formata sok (Tokenize + Pad)
new_seq = tokenizer.texts_to_sequences(random_mails)
new_pad = pad_sequences(new_seq, maxlen=max_len)

# 3. Tahmin Yap
preds = model.predict(new_pad)

# 4. Sonuçları Ekrana Yazdır
for i, mail in enumerate(random_mails):
    prob = preds[i][0]
    state = ("KATIKSIZ SPAM" if prob > 0.75 else "SPAM") if prob > 0.5 else "SECURE"
    print(f"\nMail: {mail}")
    print(f"Tahmin: {state} (Olasılık: %{prob*100:.2f})")

In [ ]:
# Modeli bir klasöre kaydet
model.save('spam_model_v2.h5')

In [ ]:
import pickle
with open('tokenizer2.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)